# 🎨 How does AI make images? A hands-on Stable Diffusion lab

*Created by **Nam Ngo** (CECS, VinUniversity) for the AI Literacy course, AI in Media session. [GitHub](https://github.com/phnam05/stable-diffusion-lab) · Free to reuse and adapt for non-commercial purposes with credit ([CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)).*

In this lab you will use a real image-generating AI, **Stable Diffusion**. It's the same kind of technology behind many AI image apps. Those apps hide how it works; here **you** turn the knobs and see what each one does.

**You don't need to know how to code.** The code is hidden. You only press ▶ buttons and change the text boxes and sliders.

### How to use this notebook
1. **Run ⚙️ 1. Setup first** (press ▶ next to its title). If Colab warns *"This notebook was not authored by Google"*, click **Run anyway**.
2. Go through the sections in order: **read → 🤔 predict → press ▶ → compare with your prediction.**
3. Change the boxes on the right of each cell to try your own ideas, then press ▶ again.

💾 To keep an image: right-click it → **Save image as…**. Nothing in this notebook is saved after you close it.

### Ground rules
- A **safety filter** is on. If an image comes out black, it was blocked, so try a different prompt.
- Only upload photos that you took yourself, and never photos of other people without their permission.
- You're borrowing a free GPU from Google, so close the tab when you're done.

| Section | You will learn… |
|---|---|
| 2 · 3 | how an image is "denoised" out of random static |
| 4 – 7 | what **seed, steps, CFG** and **negative prompt** do |
| 8 | how AI **edits** a photo (image-to-image) |
| 9 | whose picture the AI draws by default (**bias**) |
| 10 · 11 | bonus: **inpainting** and a free **playground** |

## ⚙️ 1. Setup
Press ▶ now. It downloads the AI model (about 3 GB) onto a Google computer with a GPU. This takes **1–2 minutes**, so read section 2 while you wait.

The top-right corner of Colab should show **T4**. If Setup says there is no GPU, follow its instructions.

In [ ]:
#@title ⚙️ 1. Setup: press ▶ and wait for "✅ Ready!" (about 1–2 minutes)
#@markdown Nothing to change here. Press ▶ (left of this title), then read section 2 while it loads.

# ── Instructor settings ─────────────────────────────────────────────────────
GALLERY_BASE_URL = "https://raw.githubusercontent.com/phnam05/stable-diffusion-lab/main/gallery"
MODEL_ID = "Lykon/dreamshaper-8"                     # Stable Diffusion 1.5 fine-tune (~3 GB download)
FALLBACK_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
INPAINT_MODEL_ID = "Lykon/dreamshaper-8-inpainting"
FALLBACK_INPAINT_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-inpainting"
PREVIEW_DECODER_ID = "madebyollin/taesd"            # tiny decoder for the live previews
DIFFUSERS_PIN = "diffusers==0.42.1"                 # installed only if Colab doesn't already have diffusers
# ────────────────────────────────────────────────────────────────────────────

import os, io, re, sys, time, base64, random, shutil, warnings, logging, traceback, functools, subprocess, importlib
import html as _html

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
for _n in ("diffusers", "transformers", "huggingface_hub", "accelerate", "httpx"):
    logging.getLogger(_n).setLevel(logging.ERROR)

# Hooks for offline smoke tests (not used in class).
MODEL_ID = os.environ.get("SD_LAB_MODEL", MODEL_ID)
INPAINT_MODEL_ID = os.environ.get("SD_LAB_INPAINT_MODEL", INPAINT_MODEL_ID)
_TEST = os.environ.get("SD_LAB_TEST") == "1"
_FORCE_CPU = os.environ.get("SD_LAB_FORCE_CPU") == "1"
_EXPORT_DIR = os.environ.get("SD_LAB_EXPORT_DIR", "/content/gallery")

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont, ImageOps, ImageFilter
from IPython.display import display, HTML

DEFAULTS = {'steps': 20,
 'cfg': 7.0,
 'first_prompt': 'a red fox reading a newspaper in a cozy cafe, watercolor painting',
 'first_seed': 42,
 'denoise_prompt': 'a lighthouse on a rocky cliff at sunset, oil painting',
 'denoise_seed': 7,
 'denoise_steps': 20,
 'seed_prompt': 'a cute robot watering plants in a greenhouse, 3D render',
 'seed_seeds': '1, 2, 3, 4',
 'steps_prompt': 'a bowl of ramen on a wooden table, food photography',
 'steps_seed': 3,
 'steps_list': '1, 3, 5, 10, 20, 40',
 'cfg_prompt': 'a castle made of ice on a mountain, fantasy art',
 'cfg_seed': 5,
 'cfg_list': '1, 3, 7, 12, 20',
 'neg_prompt': 'a busy street market in an old town, photo',
 'neg_negative': 'people, crowd',
 'neg_seeds': '1, 2',
 'i2i_prompt': 'a dog sitting on a park bench, oil painting, impressionist style, thick brush '
               'strokes',
 'i2i_strengths': '0.2, 0.4, 0.6, 0.8, 1.0',
 'i2i_source': 'a sample picture',
 'i2i_seed': 42,
 'i2i_sample_prompt': 'a photo of a golden retriever sitting on a park bench, sunny day',
 'i2i_sample_seed': 11,
 'bias_words': 'a CEO, a nurse, a software engineer, a housekeeper',
 'bias_per_row': 4,
 'bias_template': 'a photo of {}',
 'bias_steps': 20,
 'inp_source': 'last generated image',
 'inp_x': 30,
 'inp_y': 35,
 'inp_w': 40,
 'inp_h': 45,
 'inp_prompt': 'a giant teddy bear',
 'inp_seed': 1,
 'inp_fallback_prompt': 'a cozy living room with a sofa and a big window, photo',
 'play_prompt': 'an astronaut riding a horse on the moon, highly detailed, cinematic lighting',
 'play_negative': 'blurry, low quality, deformed',
 'play_seed': -1,
 'play_steps': 20,
 'play_cfg': 7.0,
 'play_size': '512 x 512 (square)',
 'play_sampler': 'DPM++ 2M Karras',
 'play_n': 2}
GALLERY_NAMES = {"first", "denoise", "seed", "steps", "cfg", "negative", "img2img", "bias", "inpaint"}


# ── Small display helpers ───────────────────────────────────────────────────
class LabInputError(Exception):
    """A problem with what the student typed: shown as a friendly hint, not a crash."""


def _say(msg):
    print(msg, flush=True)


def _esc(text):
    return _html.escape(str(text))


def _n_steps(n):
    return f"{n} step" if n == 1 else f"{n} steps"


def _box(html_text, bg="#fff4e5", border="#f0a020"):
    display(HTML(
        f'<div style="font-family:sans-serif;font-size:15px;line-height:1.5;padding:10px 14px;margin:8px 0;'
        f'background:{bg};border-left:5px solid {border};border-radius:6px;color:#222">{html_text}</div>'))


def _takeaway(html_text):
    _box("💡 " + html_text, "#e6f4ea", "#34a853")


def _font(size=15):
    try:
        return ImageFont.load_default(size=size)
    except TypeError:  # very old Pillow
        return ImageFont.load_default()


def _jpeg_b64(img, quality=90):
    buf = io.BytesIO()
    img.convert("RGB").save(buf, "JPEG", quality=quality)
    return base64.b64encode(buf.getvalue()).decode()


def _img_tag(img, max_px=None):
    width = f"max-width:min(100%,{max_px}px)" if max_px else "max-width:100%"
    return f'<img src="data:image/jpeg;base64,{_jpeg_b64(img)}" style="{width};height:auto;border-radius:6px">'


def _gif_bytes(frames, px=256, ms=90, hold_ms=1500):
    frames = [_fit(f, px).convert("RGB") for f in frames]
    buf = io.BytesIO()
    frames[0].save(buf, "GIF", save_all=True, append_images=frames[1:],
                   duration=[ms] * (len(frames) - 1) + [hold_ms], loop=0)
    return buf.getvalue()


class _Live:
    """A status line plus one image area that we redraw as results come in."""

    def __init__(self):
        self._s = display(HTML(""), display_id=True)
        self._v = display(HTML(""), display_id=True)

    def status(self, msg):
        if self._s is not None:
            self._s.update(HTML(
                f'<div style="display:inline-block;font-family:sans-serif;font-size:14px;background:#f1f3f4;'
                f'color:#333;padding:4px 12px;border-radius:12px;margin:4px 0">{msg}</div>' if msg else ""))

    def show(self, img, max_px=None):
        if self._v is not None:
            self._v.update(HTML(_img_tag(img, max_px)))


def _fit(img, height):
    if img.height == height:
        return img
    width = max(1, round(img.width * height / img.height))
    return img.resize((width, height), Image.LANCZOS)


def _shorten(draw, text, font, max_w):
    if draw.textlength(text, font=font) <= max_w:
        return text
    while text and draw.textlength(text + "...", font=font) > max_w:
        text = text[:-1]
    return text + "..."


def _wrap(draw, text, font, max_w):
    lines, line = [], ""
    for word in str(text).split():
        trial = (line + " " + word).strip()
        if draw.textlength(trial, font=font) <= max_w or not line:
            line = trial
        else:
            lines.append(line)
            line = word
    return lines + ([line] if line else [])


def make_strip(images, labels=None, height=256, pad=8):
    """Images side by side, each with a caption underneath."""
    ims = [_fit(im, height) for im in images]
    font = _font(15)
    cap = 26 if labels else 0
    out = Image.new("RGB", (sum(i.width for i in ims) + pad * (len(ims) + 1), height + cap + 2 * pad), "white")
    draw = ImageDraw.Draw(out)
    x = pad
    for k, im in enumerate(ims):
        out.paste(im, (x, pad))
        if labels:
            text = _shorten(draw, str(labels[k]), font, im.width - 4)
            draw.text((x + (im.width - draw.textlength(text, font=font)) / 2, pad + height + 5),
                      text, fill=(30, 30, 30), font=font)
        x += im.width + pad
    return out


def make_grid(rows, row_labels, height=192, pad=8, label_w=180):
    """One row of images per label (used by the bias experiment)."""
    font = _font(17)
    rows = [[_fit(im, height) for im in r] for r in rows]
    row_w = max(sum(i.width for i in r) + pad * len(r) for r in rows)
    out = Image.new("RGB", (label_w + row_w + pad, len(rows) * (height + pad) + pad), "white")
    draw = ImageDraw.Draw(out)
    y = pad
    for r, label in zip(rows, row_labels):
        lines = _wrap(draw, label, font, label_w - 2 * pad)
        ty = y + height / 2 - len(lines) * 11
        for ln in lines:
            draw.text((pad, ty), ln, fill=(30, 30, 30), font=font)
            ty += 22
        x = label_w
        for im in r:
            out.paste(im, (x, y))
            x += im.width + pad
        y += height + pad
    return out


def _overlay(img, box):
    """Show the area to repaint as a see-through red box."""
    out = img.convert("RGBA")
    layer = Image.new("RGBA", out.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(layer)
    d.rectangle(box, fill=(255, 40, 40, 110), outline=(255, 0, 0, 255), width=max(1, out.width // 170))
    return Image.alpha_composite(out, layer).convert("RGB")


def rect_mask(size, x, y, w, h):
    W, H = size
    x0, y0 = int(W * float(x) / 100), int(H * float(y) / 100)
    x1, y1 = min(W, x0 + int(W * float(w) / 100)), min(H, y0 + int(H * float(h) / 100))
    if x1 - x0 < 16 or y1 - y0 < 16:
        raise LabInputError("The red box is too small (or pushed off the image). Make width/height bigger or move x/y.")
    mask = Image.new("L", size, 0)
    ImageDraw.Draw(mask).rectangle([x0, y0, x1, y1], fill=255)
    return mask, (x0, y0, x1, y1)


def _numbers(text, kind, lo, hi, max_n, what):
    parts = [p for p in re.split(r"[,;\s]+", str(text).strip()) if p]
    if not parts:
        raise LabInputError(f"Please type at least one {what}, separated by commas, like: 1, 2, 3")
    vals, clamped = [], False
    for p in parts:
        try:
            v = float(p)
        except ValueError:
            raise LabInputError(f'"{_esc(p)}" is not a number. Type {what}s separated by commas, like: 1, 2, 3')
        if v < lo or v > hi:
            clamped = True
            v = min(max(v, lo), hi)
        vals.append(int(round(v)) if kind is int else round(v, 2))
    if clamped:
        _say(f"(Some values were changed to stay between {lo:g} and {hi:g}.)")
    if len(vals) > max_n:
        _say(f"(Using only the first {max_n} values to keep it quick.)")
        vals = vals[:max_n]
    return vals


def _prep(img, long_side=512):
    """Rotate phone photos upright and resize so the long side is 512 (multiples of 64)."""
    if _TEST:
        long_side = 64
    img = ImageOps.exif_transpose(img).convert("RGB")
    s = long_side / max(img.size)
    snap = lambda v: max(64, int(round(v * s / 64)) * 64)
    return img.resize((snap(img.width), snap(img.height)), Image.LANCZOS)


def _upload_photo():
    try:
        from google.colab import files
    except ImportError:
        raise LabInputError("Uploading only works inside Google Colab.")
    _say("👇 Click 'Choose Files' (or 'Browse') below and pick a JPG or PNG photo.")
    uploaded = files.upload()
    if not uploaded:
        raise LabInputError("No photo was uploaded. Run the cell again and choose a file.")
    name, data = next(iter(uploaded.items()))
    try:
        if os.path.exists(name):
            os.remove(name)  # keep no copy on the Colab machine
    except OSError:
        pass
    try:
        img = Image.open(io.BytesIO(data))
        img.load()
    except Exception:
        raise LabInputError(f'Could not open "{_esc(name)}". Please upload a JPG or PNG. '
                            "(iPhone HEIC photos don't work: take a screenshot of the photo and upload that.)")
    return img


# ── Model loading ───────────────────────────────────────────────────────────
def _ensure_diffusers():
    try:
        import diffusers
    except ImportError:
        _say("     Installing diffusers (one time, ~20 s)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", DIFFUSERS_PIN], check=True)
        importlib.invalidate_caches()
        import diffusers
    try:  # keep the output readable: no progress bars, no "unauthenticated requests" notices
        from diffusers.utils import logging as _dlog
        _dlog.set_verbosity_error()
        _dlog.disable_progress_bar()
        import transformers
        transformers.logging.set_verbosity_error()
        from huggingface_hub.utils import logging as _hlog
        _hlog.set_verbosity_error()
    except Exception:
        pass
    return diffusers.__version__


def _load_pipeline(cls, repo_ids, what):
    last = None
    for repo in repo_ids:
        tries = ([dict(variant="fp16", use_safetensors=True)] if _HAS_GPU else []) + [dict(use_safetensors=True), dict()]
        for kw in tries:
            try:
                pipe = cls.from_pretrained(repo, torch_dtype=_DTYPE, **kw)
                if repo != repo_ids[0]:
                    _say(f"     (The main {what} was unavailable, so the backup {repo} is used.)")
                return pipe, repo
            except Exception as e:
                last = e
    raise RuntimeError(f"Could not download the {what}. Check the internet connection and run Setup again. "
                       f"({type(last).__name__}: {str(last)[:200]})")


def _make_scheduler(name, base_config):
    from diffusers import (DPMSolverMultistepScheduler, EulerAncestralDiscreteScheduler,
                           EulerDiscreteScheduler, DDIMScheduler, DEISMultistepScheduler)
    if name.startswith("DPM++"):
        return DPMSolverMultistepScheduler.from_config(
            base_config, algorithm_type="dpmsolver++", solver_type="midpoint", use_karras_sigmas=True)
    if name == "Euler a":
        return EulerAncestralDiscreteScheduler.from_config(base_config)
    if name == "Euler":
        return EulerDiscreteScheduler.from_config(base_config)
    if name == "DDIM":
        return DDIMScheduler.from_config(base_config)
    return DEISMultistepScheduler.from_config(base_config)


# ── The lab: one method per notebook section ────────────────────────────────
def _lab_section(name):
    """Wraps a section: shows the example gallery when there is no GPU, and turns errors into plain English."""
    def deco(fn):
        @functools.wraps(fn)
        def wrapper(self, *args, **kwargs):
            if self.fallback:
                self.show_fallback(name)
                return None
            try:
                fn(self, *args, **kwargs)
            except LabInputError as e:
                _box(f"✏️ {e}", "#e8f0fe", "#4a7bd0")
            except KeyboardInterrupt:
                _say("⏹️ Stopped.")
            except Exception as e:
                self.last_error = traceback.format_exc()
                text = str(e)
                if "out of memory" in text.lower():
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    _box("😵 The GPU ran out of memory. Try a smaller size or fewer images. If it keeps happening: "
                         "menu <b>Runtime ▸ Restart session</b>, then run ⚙️ Setup again.", "#fdecea", "#d93025")
                else:
                    first = (text.strip().splitlines() or [""])[0][:300]
                    _box(f"⚠️ Something went wrong: <code>{_esc(type(e).__name__)}: {_esc(first)}</code><br>"
                         "Try running this cell again. If that doesn't help, run ⚙️ Setup again or ask the instructor. "
                         "<span style='color:#666'>(Instructor: <code>print(lab.last_error)</code> shows details.)</span>",
                         "#fdecea", "#d93025")
            return None
        return wrapper
    return deco


class _Lab:
    def __init__(self, fallback):
        self.fallback = fallback
        self.pipe = self.taesd = self._i2i = self._inpaint = None
        self.model_id = None
        self.results = {}
        self.last_image = None
        self.photo = None
        self._sample = None
        self._inpaint_job = None
        self.last_error = None

    # ---------- setup ----------
    def load(self):
        from diffusers import StableDiffusionPipeline, AutoencoderTiny
        _say("2/4  Downloading + loading the image model (~3 GB, about 1 minute)...")
        t = time.time()
        pipe, self.model_id = _load_pipeline(StableDiffusionPipeline, [MODEL_ID, FALLBACK_MODEL_ID], "image model")
        self.base_scheduler_config = pipe.scheduler.config
        pipe.scheduler = _make_scheduler("DPM++ 2M Karras", self.base_scheduler_config)
        pipe.set_progress_bar_config(disable=True)
        self.pipe = pipe.to(_DEVICE)
        _say(f"     ✓ {self.model_id} loaded in {time.time() - t:.0f} s")
        _say("3/4  Loading the live-preview decoder...")
        try:
            self.taesd = AutoencoderTiny.from_pretrained(PREVIEW_DECODER_ID, torch_dtype=_DTYPE).to(_DEVICE)
        except Exception:
            self.taesd = None
            _say("     (Preview decoder unavailable: previews will use the main decoder, a bit slower.)")
        _say("4/4  Warming up the GPU...")
        self.txt2img("warm-up", seed=0, steps=2)

    # ---------- core generation ----------
    def _clean(self, steps, cfg, width, height, max_side=768):
        steps = int(min(max(int(steps), 1), 50))
        cfg = float(min(max(float(cfg), 0.0), 30.0))
        snap = lambda v: int(min(max(round(int(v) / 64) * 64, 256), max_side))
        width, height = snap(width), snap(height)
        if _TEST:
            width = height = 64
            steps = min(steps, 3)
        return steps, cfg, width, height

    @staticmethod
    def _gens(seeds):
        return [torch.Generator("cpu").manual_seed(int(s) % (2 ** 32)) for s in seeds]

    @staticmethod
    def _flags(out, n):
        f = getattr(out, "nsfw_content_detected", None)
        return [bool(x) for x in f] if f is not None else [False] * n

    def txt2img(self, prompt, seed=42, steps=DEFAULTS["steps"], cfg=DEFAULTS["cfg"], negative="",
                width=512, height=512, seeds=None, max_side=768, scheduler=None, **extra):
        steps, cfg, width, height = self._clean(steps, cfg, width, height, max_side)
        seeds = list(seeds) if seeds else [seed]
        gens = self._gens(seeds)
        old = self.pipe.scheduler
        if scheduler:
            self.pipe.scheduler = _make_scheduler(scheduler, self.base_scheduler_config)
        t0 = time.time()
        try:
            out = self.pipe(prompt=str(prompt), negative_prompt=(str(negative) if negative else None),
                            num_inference_steps=steps, guidance_scale=cfg, width=width, height=height,
                            generator=gens if len(gens) > 1 else gens[0], num_images_per_prompt=len(gens), **extra)
        finally:
            self.pipe.scheduler = old
        return out.images, self._flags(out, len(out.images)), time.time() - t0

    @property
    def i2i(self):
        if self._i2i is None:
            from diffusers import AutoPipelineForImage2Image
            self._i2i = AutoPipelineForImage2Image.from_pipe(self.pipe)
            self._i2i.set_progress_bar_config(disable=True)
        return self._i2i

    def _get_inpaint(self, live):
        if self._inpaint is None:
            from diffusers import StableDiffusionInpaintPipeline
            live.status("Loading the inpainting model (one-time download, about 1 minute)...")
            pipe, _ = _load_pipeline(StableDiffusionInpaintPipeline,
                                     [INPAINT_MODEL_ID, FALLBACK_INPAINT_MODEL_ID], "inpainting model")
            pipe.scheduler = _make_scheduler("DPM++ 2M Karras", pipe.scheduler.config)
            pipe.set_progress_bar_config(disable=True)
            self._inpaint = pipe.to(_DEVICE)
        return self._inpaint

    @torch.no_grad()
    def _preview(self, latents, px=256):
        """Turn the model's internal (compressed) picture into a small viewable image."""
        lat = latents[:1]
        try:
            if self.taesd is None:
                raise RuntimeError("no preview decoder")
            x = self.taesd.decode(lat.to(self.taesd.dtype)).sample
        except Exception:
            vae = self.pipe.vae
            x = vae.decode((lat / vae.config.scaling_factor).to(vae.dtype)).sample
        x = ((x[0].float().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()
        return _fit(Image.fromarray((x * 255).round().astype("uint8")), px)

    def _source_image(self, source):
        s = str(source)
        if s.startswith("upload"):
            self.photo = _prep(_upload_photo())
            return self.photo
        if s.startswith("my photo"):
            if self.photo is None:
                self.photo = _prep(_upload_photo())
            return self.photo
        if s.startswith("last"):
            if self.last_image is None:
                _say("(You haven't made an image in section 2 or the playground yet, so here's one to work with.)")
                (self.last_image,), _, _ = self.txt2img(DEFAULTS["inp_fallback_prompt"], seed=3)
            return _prep(self.last_image)
        if self._sample is None:
            (self._sample,), _, _ = self.txt2img(DEFAULTS["i2i_sample_prompt"], seed=DEFAULTS["i2i_sample_seed"])
        return _prep(self._sample)

    @staticmethod
    def _blocked_note(flags):
        k = sum(bool(f) for f in flags)
        if k:
            _box(f"🛡️ The safety filter blocked {k} image{'s' if k > 1 else ''} (shown in black). Try another prompt "
                 "or seed. AI apps use filters like this to moderate content, and they sometimes block harmless "
                 "images too.", "#f3e8fd", "#9334e6")

    @staticmethod
    def _label(text, blocked):
        return f"{text} [blocked]" if blocked else text

    def show_fallback(self, name):
        if name not in GALLERY_NAMES:
            _box("🖥️ This part needs a GPU, and this notebook doesn't have one. Watch a teammate's screen, or try "
                 "<b>Runtime ▸ Change runtime type ▸ T4 GPU</b> and run ⚙️ Setup again.")
            return
        if not GALLERY_BASE_URL or "YOUR-GITHUB-USERNAME" in GALLERY_BASE_URL:
            _box("🖥️ No GPU in this notebook, so this section can't run here. Follow along on a teammate's screen.")
            return
        _box("🖥️ No GPU in this notebook, so here is an <b>example result</b> made with the default settings. "
             "Make your prediction first, then compare!")
        files = [f"{name}.jpg"] + (["denoise.gif"] if name == "denoise" else [])
        display(HTML("".join(f'<img src="{GALLERY_BASE_URL}/{f}" style="max-width:100%;height:auto;'
                             f'border-radius:6px;margin:4px 8px 4px 0">' for f in files)))

    # ---------- sections ----------
    @_lab_section("first")
    def run_first(self, prompt, seed):
        live = _Live()
        live.status("🎨 Painting...")
        (img,), (blocked,), dt = self.txt2img(prompt, seed=seed)
        live.show(img, 512)
        live.status(f"seed {int(seed)} · {DEFAULTS['steps']} steps · CFG {DEFAULTS['cfg']:g} · made in {dt:.1f} s")
        self._blocked_note([blocked])
        self.last_image = img
        self.results["first"] = img

    @_lab_section("denoise")
    def run_denoise(self, prompt, seed, steps=DEFAULTS["denoise_steps"]):
        from diffusers.utils.torch_utils import randn_tensor
        steps, _, w, h = self._clean(steps, DEFAULTS["cfg"], 512, 512)
        p = self.pipe
        live = _Live()
        shape = (1, p.unet.config.in_channels, h // p.vae_scale_factor, w // p.vae_scale_factor)
        noise = randn_tensor(shape, generator=torch.Generator("cpu").manual_seed(int(seed)),
                             device=torch.device(_DEVICE), dtype=p.unet.dtype)
        frames = [self._preview(noise)]
        live.show(frames[0], 256)
        live.status("Step 0: pure random noise")

        def on_step(pipe, i, t, kw):
            frames.append(self._preview(kw["latents"]))
            live.show(frames[-1], 256)
            live.status(f"Removing noise... step {i + 1} of {steps}")
            return kw

        (img,), (blocked,), dt = self.txt2img(prompt, seed=seed, steps=steps, callback_on_step_end=on_step)
        idx = sorted({round(k * (len(frames) - 1) / 7) for k in range(8)})
        strip = make_strip([frames[k] for k in idx], ["noise" if k == 0 else f"step {k}" for k in idx], height=192)
        live.show(strip)
        live.status(f"Done in {dt:.1f} s. Top: snapshots along the way. Below: replay + final image.")
        gif = _gif_bytes(frames + [img])
        display(HTML(
            '<div style="display:flex;gap:16px;flex-wrap:wrap;font-family:sans-serif;font-size:13px;color:#888">'
            f'<figure style="margin:0"><img src="data:image/gif;base64,{base64.b64encode(gif).decode()}" '
            'style="width:256px;border-radius:6px"><figcaption>Replay (loops)</figcaption></figure>'
            f'<figure style="margin:0">{_img_tag(img, 256)}<figcaption>Final image</figcaption></figure></div>'))
        self._blocked_note([blocked])
        self.results["denoise"] = strip
        self.results["denoise.gif"] = gif
        _takeaway("The picture is not pasted in from somewhere: it is <b>sculpted out of random noise</b>, a little at a "
                  "time. The rough layout (big shapes, colors) appears in the first few steps; later steps only add detail.")

    @_lab_section("seed")
    def run_seed(self, prompt, seeds):
        seeds = _numbers(seeds, int, 0, 2 ** 32 - 1, 6, "seed number")
        live = _Live()
        imgs, labels, flags = [], [], []
        n = len(seeds) + 1
        for k, s in enumerate(seeds):
            live.status(f"Making image {k + 1} of {n}...")
            (im,), (bl,), _ = self.txt2img(prompt, seed=s)
            imgs.append(im)
            labels.append(self._label(f"seed {s}", bl))
            flags.append(bl)
            live.show(make_strip(imgs, labels))
        live.status(f"Making image {n} of {n}: seed {seeds[0]} again...")
        (again,), (bl,), _ = self.txt2img(prompt, seed=seeds[0])
        diff = float(np.abs(np.asarray(imgs[0], np.int16) - np.asarray(again, np.int16)).mean() / 255 * 100)
        imgs.append(again)
        labels.append(self._label(f"seed {seeds[0]} AGAIN", bl))
        flags.append(bl)
        strip = make_strip(imgs, labels)
        live.show(strip)
        live.status("")
        self._blocked_note(flags)
        self.results["seed"] = strip
        same = "exactly the same" if diff < 0.5 else "almost exactly the same"
        _takeaway(f"Different seeds → different pictures of the same idea. Running seed {seeds[0]} again gave "
                  f"<b>{same}</b> image (pixel difference: {diff:.2f}%). That's why people share their seed: it lets "
                  "anyone recreate an image exactly.")

    @_lab_section("steps")
    def run_steps(self, prompt, seed, steps_to_compare):
        vals = _numbers(steps_to_compare, int, 1, 50, 6, "step count")
        live = _Live()
        imgs, labels, flags = [], [], []
        for k, v in enumerate(vals):
            live.status(f"Making image {k + 1} of {len(vals)} ({_n_steps(v)})...")
            (im,), (bl,), dt = self.txt2img(prompt, seed=seed, steps=v)
            imgs.append(im)
            labels.append(self._label(f"{_n_steps(v)} · {dt:.1f}s", bl))
            flags.append(bl)
            live.show(make_strip(imgs, labels))
        live.status("")
        self._blocked_note(flags)
        self.results["steps"] = make_strip(imgs, labels)
        _takeaway("Too few steps → blurry and unfinished. After about 20 steps the image barely changes, but the time "
                  "keeps growing. More is not always better: apps choose this trade-off between quality, speed "
                  "and cost (energy!) for you.")

    @_lab_section("cfg")
    def run_cfg(self, prompt, seed, cfg_to_compare):
        vals = _numbers(cfg_to_compare, float, 1, 30, 6, "CFG value")
        live = _Live()
        imgs, labels, flags = [], [], []
        for k, v in enumerate(vals):
            live.status(f"Making image {k + 1} of {len(vals)} (CFG {v:g})...")
            (im,), (bl,), _ = self.txt2img(prompt, seed=seed, cfg=v)
            imgs.append(im)
            labels.append(self._label(f"CFG {v:g}", bl))
            flags.append(bl)
            live.show(make_strip(imgs, labels))
        live.status("")
        self._blocked_note(flags)
        self.results["cfg"] = make_strip(imgs, labels)
        _takeaway("Low CFG: the prompt is only a hint, so the image is washed-out and dreamy. Around 7: balanced. Very "
                  "high: the image gets 'fried', with harsh colors and contrast. Most apps hide this knob and pick a "
                  "middle value for you.")

    @_lab_section("negative")
    def run_negative(self, prompt, negative_prompt, seeds):
        if not str(negative_prompt).strip():
            raise LabInputError("Type something in <b>negative_prompt</b>: the things you DON'T want to see.")
        seeds = _numbers(seeds, int, 0, 2 ** 32 - 1, 3, "seed number")
        live = _Live()
        imgs, labels, flags = [], [], []
        total = 2 * len(seeds)
        for k, s in enumerate(seeds):
            for j, neg in enumerate(("", negative_prompt)):
                live.status(f"Making image {2 * k + j + 1} of {total}...")
                (im,), (bl,), _ = self.txt2img(prompt, seed=s, negative=neg)
                imgs.append(im)
                labels.append(self._label(f"seed {s} · {'with' if neg else 'no'} negative", bl))
                flags.append(bl)
                live.show(make_strip(imgs, labels))
        live.status("")
        self._blocked_note(flags)
        self.results["negative"] = make_strip(imgs, labels)
        _takeaway(f'The negative prompt (<i>"{_esc(negative_prompt)}"</i>) pushed the AI <b>away</b> from those things. '
                  "It works through the same trick as CFG: the 'no-prompt' guess is swapped for your negative prompt.")

    @_lab_section("img2img")
    def run_img2img(self, prompt, strengths, source, seed=DEFAULTS["i2i_seed"]):
        vals = _numbers(strengths, float, 0.1, 1.0, 6, "strength")
        src = self._source_image(source)
        live = _Live()
        imgs, labels, flags = [src], ["original"], []
        live.show(make_strip(imgs, labels))
        steps, cfg, _, _ = self._clean(DEFAULTS["steps"], DEFAULTS["cfg"], 512, 512)
        for k, s in enumerate(vals):
            live.status(f"Repainting with strength {s:g} ({k + 1} of {len(vals)})...")
            n_steps = max(steps, int(np.ceil(1 / s)))
            out = self.i2i(prompt=str(prompt), image=src, strength=s, num_inference_steps=n_steps,
                           guidance_scale=cfg, generator=self._gens([seed])[0])
            bl = self._flags(out, 1)[0]
            imgs.append(out.images[0])
            labels.append(self._label(f"strength {s:g} · {_n_steps(max(1, int(n_steps * s)))}", bl))
            flags.append(bl)
            live.show(make_strip(imgs, labels))
        live.status("")
        self._blocked_note(flags)
        self.results["img2img"] = make_strip(imgs, labels)
        _takeaway("Low strength: only a little noise is added, so the AI just touches up the photo. High strength: the "
                  "photo is buried in noise and the AI repaints it from the prompt. At 1.0 the original is completely "
                  "gone. (This is also why AI 'editing' can quietly change things you didn't ask it to change.)")

    @_lab_section("bias")
    def run_bias(self, things_to_test, images_per_row=DEFAULTS["bias_per_row"]):
        items = [w.strip() for w in re.split(r"[,\n;]+", str(things_to_test)) if w.strip()]
        if not items:
            raise LabInputError("Type at least one thing to test, like: a doctor, a nurse")
        if len(items) > 5:
            _say("(Using only the first 5 items to keep it quick.)")
            items = items[:5]
        per_row = int(min(max(int(images_per_row), 2), 6))
        seeds = list(range(1, per_row + 1))
        template = DEFAULTS["bias_template"]
        live = _Live()
        rows, flags = [], []
        for k, w in enumerate(items):
            live.status(f'Drawing "{_esc(w)}" ({k + 1} of {len(items)})...')
            imgs, fl, _ = self.txt2img(template.format(w), seeds=seeds, steps=DEFAULTS["bias_steps"])
            rows.append(imgs)
            flags += fl
            live.show(make_grid(rows, items[:len(rows)]))
        grid = make_grid(rows, items)
        live.status(f'Prompt: "{_esc(template.format("…"))}" · same seeds ({seeds[0]}–{seeds[-1]}) for every row')
        self._blocked_note(flags)
        self.results["bias"] = grid
        _takeaway("Before discussing, look closely: gender, age, skin tone, body type, clothes, setting, facial "
                  "expression. Then scroll up to the 💬 <b>Discuss</b> questions.")

    @_lab_section("inpaint_area")
    def run_inpaint_area(self, source, x, y, width, height):
        src = self._source_image(source)
        mask, box = rect_mask(src.size, x, y, width, height)
        self._inpaint_job = (src, mask, box)
        display(HTML(_img_tag(_overlay(src, box), 512)))
        _say("👍 Happy with the red box? Go to Step B. Not happy? Move the sliders and press ▶ again.")

    @_lab_section("inpaint")
    def run_inpaint(self, prompt, seed):
        if self._inpaint_job is None:
            raise LabInputError("First run <b>Step A</b> (choose the area) just above.")
        src, mask, box = self._inpaint_job
        live = _Live()
        pipe = self._get_inpaint(live)
        steps, cfg, _, _ = self._clean(DEFAULTS["steps"], 7.5, 512, 512)
        imgs, labels, flags = [src, _overlay(src, box)], ["original", "area to repaint"], []
        live.show(make_strip(imgs, labels))
        for k, s in enumerate((int(seed), int(seed) + 1)):
            live.status(f"Repainting the red area ({k + 1} of 2)...")
            out = pipe(prompt=str(prompt), image=src, mask_image=mask, width=src.width, height=src.height,
                       num_inference_steps=steps, guidance_scale=cfg, generator=self._gens([s])[0])
            bl = self._flags(out, 1)[0]
            result = out.images[0].resize(src.size)
            if not bl:  # paste the untouched pixels back, with a soft edge, so only the box really changes
                result = Image.composite(result, src, mask.filter(ImageFilter.GaussianBlur(4)))
            imgs.append(result)
            labels.append(self._label(f"result · seed {s}", bl))
            flags.append(bl)
            live.show(make_strip(imgs, labels))
        live.status("")
        self._blocked_note(flags)
        self.results["inpaint"] = make_strip(imgs, labels)
        _takeaway("Only the boxed area was repainted, and the rest of the picture stayed the same. The new object usually "
                  "matches the light and colors around it, because the AI 'sees' the whole picture while it denoises "
                  "the box.")

    @_lab_section("playground")
    def run_playground(self, prompt, negative_prompt, seed, steps, cfg, size, sampler, how_many):
        m = re.match(r"\s*(\d+)\s*x\s*(\d+)", str(size))
        width, height = (int(m[1]), int(m[2])) if m else (512, 512)
        seed = int(seed)
        if seed < 0:
            seed = random.randint(0, 2 ** 31 - 1)
        n = int(min(max(int(how_many), 1), 4))
        if width * height > 768 * 768 and n > 2:
            _say("(Big images are slow, so we'll make 2.)")
            n = 2
        seeds = [seed + k for k in range(n)]
        live = _Live()
        live.status(f"🎨 Making {n} image{'s' if n > 1 else ''}...")
        imgs, flags, dt = self.txt2img(prompt, seeds=seeds, steps=steps, cfg=cfg, negative=negative_prompt,
                                       width=width, height=height, max_side=1024, scheduler=sampler)
        live.show(make_strip(imgs, [self._label(f"seed {s}", b) for s, b in zip(seeds, flags)],
                             height=384 if n <= 2 else 256))
        live.status(f"{_esc(sampler)} · {_n_steps(int(steps))} · CFG {float(cfg):g} · {width}×{height} · {dt:.1f} s "
                    f"· seeds {seeds[0]}{'–' + str(seeds[-1]) if n > 1 else ''}")
        self._blocked_note(flags)
        self.last_image = imgs[0]

    # ---------- instructor ----------
    def export_gallery(self):
        """Re-run every experiment with its default settings and save the results for the no-GPU fallback."""
        if self.fallback:
            _say("The gallery export needs a GPU runtime.")
            return
        d = DEFAULTS
        self.results = {}
        self.run_first(d["first_prompt"], d["first_seed"])
        self.run_denoise(d["denoise_prompt"], d["denoise_seed"], d["denoise_steps"])
        self.run_seed(d["seed_prompt"], d["seed_seeds"])
        self.run_steps(d["steps_prompt"], d["steps_seed"], d["steps_list"])
        self.run_cfg(d["cfg_prompt"], d["cfg_seed"], d["cfg_list"])
        self.run_negative(d["neg_prompt"], d["neg_negative"], d["neg_seeds"])
        self.run_img2img(d["i2i_prompt"], d["i2i_strengths"], d["i2i_source"], d["i2i_seed"])
        self.run_bias(d["bias_words"], d["bias_per_row"])
        self.last_image = self.results.get("first")
        self.run_inpaint_area(d["inp_source"], d["inp_x"], d["inp_y"], d["inp_w"], d["inp_h"])
        self.run_inpaint(d["inp_prompt"], d["inp_seed"])
        shutil.rmtree(_EXPORT_DIR, ignore_errors=True)
        os.makedirs(_EXPORT_DIR, exist_ok=True)
        for name, val in self.results.items():
            path = os.path.join(_EXPORT_DIR, name if "." in name else f"{name}.jpg")
            if isinstance(val, bytes):
                with open(path, "wb") as f:
                    f.write(val)
            else:
                val.convert("RGB").save(path, "JPEG", quality=88)
        missing = sorted(GALLERY_NAMES - set(self.results))
        archive = shutil.make_archive(_EXPORT_DIR, "zip", _EXPORT_DIR)
        _say(f"Saved {len(self.results)} files to {archive}" + (f" (missing: {', '.join(missing)})" if missing else ""))
        try:
            from google.colab import files
            files.download(archive)
        except ImportError:
            pass


# ── Run setup ───────────────────────────────────────────────────────────────
if "lab" in globals() and getattr(lab, "pipe", None) is not None:
    _box("✅ Already set up: the model is loaded. Continue with section 2.", "#e6f4ea", "#34a853")
else:
    _say("1/4  Checking for a GPU...")
    _HAS_GPU = torch.cuda.is_available()
    _DEVICE = "cuda" if _HAS_GPU else "cpu"
    _DTYPE = torch.float16 if _HAS_GPU else torch.float32
    if _HAS_GPU:
        _say(f"     ✓ GPU found: {torch.cuda.get_device_name(0)}")
    if not _HAS_GPU and not _FORCE_CPU:
        lab = _Lab(fallback=True)
        _box("⚠️ <b>No GPU connected.</b> Image generation needs one.<br>"
             "1. Menu <b>Runtime ▸ Change runtime type</b> ▸ choose <b>T4 GPU</b> ▸ <b>Save</b>.<br>"
             "2. Run this Setup cell again.<br>"
             "If Colab says no GPU is available right now: no problem! Team up with a neighbour whose notebook works. "
             "You can still run the cells below to see <b>example results</b>.")
    else:
        try:
            _say(f"     ✓ diffusers {_ensure_diffusers()}")
            _new_lab = _Lab(fallback=False)
            _new_lab.load()
            lab = _new_lab
            _box("✅ <b>Ready!</b> Scroll down to section 2 and make your first image.", "#e6f4ea", "#34a853")
        except Exception as _e:
            _box(f"❌ <b>Setup failed:</b> <code>{_esc(str(_e)[:400])}</code><br>Try: menu <b>Runtime ▸ Restart "
                 "session</b>, then run this cell again. If it fails twice, team up with a neighbour.",
                 "#fdecea", "#d93025")

## 🎨 2. Your first AI image
Type a description (a **prompt**) and press ▶. Good prompts say:
- **what** (the subject): *a red fox reading a newspaper*
- **where** (the setting): *in a cozy café*
- **how** (the style): *watercolor painting*, *photo*, *3D render*, *pixel art*, *pencil sketch*…

Ideas: *a futuristic city made of candy, 3D render* · *a cat astronaut floating in space, photo* · *your campus in the year 3000, concept art*

In [ ]:
#@title ▶ Make my first image
prompt = "a red fox reading a newspaper in a cozy cafe, watercolor painting" #@param {type:"string"}
seed = 42 #@param {type:"integer"}
if "lab" not in globals():
    print("⚠️ Please run ⚙️ 1. Setup first (scroll up), then run this cell again.")
else:
    lab.run_first(prompt, seed)

## 🔍 3. Watch the AI "denoise"
This is the big idea behind Stable Diffusion:

1. **Training.** The model was shown **hundreds of millions of pictures with captions**, collected from the internet. Each picture was buried in random noise (like TV static), a little more each time, and the model practiced **guessing the noise so it could remove it**, using the caption as a hint.
2. **Generating.** To make a *new* picture, we start from **pure static** and ask the model, again and again: *"remove a bit of noise so this looks more like: <your prompt>"*. After about 20 small clean-ups, a picture appears.

So the AI doesn't copy and paste a picture: it **sculpts one out of noise**. (To be fast, it works on a small compressed "sketch" of the image, 64×64 instead of 512×512. A decoder turns that sketch into pixels.)

🤔 **Predict first:** after only 3 of 20 steps, will you see a blurry version of the final image, or still only noise?

In [ ]:
#@title ▶ Watch it denoise, step by step
prompt = "a lighthouse on a rocky cliff at sunset, oil painting" #@param {type:"string"}
seed = 7 #@param {type:"integer"}
steps = 20 #@param {type:"slider", min:5, max:50, step:1}
if "lab" not in globals():
    print("⚠️ Please run ⚙️ 1. Setup first (scroll up), then run this cell again.")
else:
    lab.run_denoise(prompt, seed, steps)

## 🎲 4. Seed: which static do we start from?
The **seed** is just a number that decides *which* random static the AI starts from.
- Same prompt + same seed + same settings → **exactly the same image**, on any computer.
- Different seed → a different picture of the same idea.

Think of shuffling a deck of cards: the seed is which shuffle you got.

🤔 **Predict first:** when we run the first seed a second time, will the image be slightly different or identical?

In [ ]:
#@title ▶ Compare seeds
prompt = "a cute robot watering plants in a greenhouse, 3D render" #@param {type:"string"}
seeds = "1, 2, 3, 4" #@param {type:"string"}
#@markdown Type up to 6 seed numbers separated by commas (any whole numbers you like).
if "lab" not in globals():
    print("⚠️ Please run ⚙️ 1. Setup first (scroll up), then run this cell again.")
else:
    lab.run_seed(prompt, seeds)

## 👣 5. Steps: how many clean-ups?
Each **step** removes some noise. More steps mean more chances to refine the details, but every step costs time (and electricity).

🤔 **Predict first:** is 40 steps twice as good as 20? Where do you think the image stops getting better?

In [ ]:
#@title ▶ Compare step counts
prompt = "a bowl of ramen on a wooden table, food photography" #@param {type:"string"}
seed = 3 #@param {type:"integer"}
steps_to_compare = "1, 3, 5, 10, 20, 40" #@param {type:"string"}
#@markdown Step counts between 1 and 50, separated by commas. The time each image took is shown under it.
if "lab" not in globals():
    print("⚠️ Please run ⚙️ 1. Setup first (scroll up), then run this cell again.")
else:
    lab.run_steps(prompt, seed, steps_to_compare)

## 🧭 6. CFG (guidance scale): how strongly to follow the prompt
At every step the AI actually makes **two guesses**: one *with* your prompt and one *without any prompt*. **CFG** decides how far to push from the "no-prompt" guess toward the "with-prompt" guess.
- **Low (1–3):** the prompt is only a gentle hint.
- **Medium (5–9):** balanced, the usual default.
- **High (15+):** pushed too hard.

It's like seasoning: some is good, too much ruins the dish.

🤔 **Predict first:** which value will match your prompt best? Which will look most natural? What happens at 20?

In [ ]:
#@title ▶ Compare CFG values
prompt = "a castle made of ice on a mountain, fantasy art" #@param {type:"string"}
seed = 5 #@param {type:"integer"}
cfg_to_compare = "1, 3, 7, 12, 20" #@param {type:"string"}
#@markdown CFG values between 1 and 30, separated by commas.
if "lab" not in globals():
    print("⚠️ Please run ⚙️ 1. Setup first (scroll up), then run this cell again.")
else:
    lab.run_cfg(prompt, seed, cfg_to_compare)

## 🚫 7. Negative prompt: what you DON'T want
Remember the "no-prompt" guess from CFG? A **negative prompt** replaces it with a description of things to avoid, so the AI pushes **away** from them.
People use it to remove things (*"cars, people"*) or to avoid common flaws (*"blurry, extra fingers, low quality"*).

🤔 **Predict first:** the prompt is a busy street market and the negative prompt is "people, crowd". What will change?

In [ ]:
#@title ▶ Compare with and without a negative prompt
prompt = "a busy street market in an old town, photo" #@param {type:"string"}
negative_prompt = "people, crowd" #@param {type:"string"}
seeds = "1, 2" #@param {type:"string"}
if "lab" not in globals():
    print("⚠️ Please run ⚙️ 1. Setup first (scroll up), then run this cell again.")
else:
    lab.run_negative(prompt, negative_prompt, seeds)

## 🖼️ 8. Image-to-image: AI editing a photo
Instead of starting from pure static, we start from **a photo with some static added**. The **strength** decides how much static:
- **Low strength (0.2–0.4):** a little noise, so the AI only touches up and the photo stays recognizable.
- **High strength (0.8–1.0):** the photo is mostly buried, so the AI repaints almost everything.

Strength also decides how many steps actually run: 0.4 × 20 steps = 8 steps.

Use the sample picture, or set **source** to *my photo* to upload one of your own (a landscape, your pet, your desk…). **With your own photo, change the prompt to describe what's in it**, plus the style you want.

🔒 Your photo is only used inside your own Colab session and disappears when you close it. Don't upload photos of other people without their permission.

🤔 **Predict first:** at which strength will the original stop being recognizable?

In [ ]:
#@title ▶ Restyle a picture
prompt = "a dog sitting on a park bench, oil painting, impressionist style, thick brush strokes" #@param {type:"string"}
source = "a sample picture" #@param ["a sample picture", "my photo (asks to upload the first time)", "upload a new photo"]
strengths = "0.2, 0.4, 0.6, 0.8, 1.0" #@param {type:"string"}
#@markdown Strengths between 0.1 and 1.0, separated by commas.
if "lab" not in globals():
    print("⚠️ Please run ⚙️ 1. Setup first (scroll up), then run this cell again.")
else:
    lab.run_img2img(prompt, strengths, source)

## ⚖️ 9. Whose picture is the "default"? A bias experiment
The AI learned from pictures and captions collected from the internet, so it also learned the internet's **patterns and stereotypes**. Let's test it. We ask for different people and things, with the **same seeds for every row**, and look at what the AI draws when we don't specify anything else.

🤔 **Predict first:** for *a CEO* and *a nurse*, what gender, age and skin tone do you expect the AI to show? Write down your guess, then run the cell.

Try the other presets in the dropdown, or type your own list (separate items with commas).

### 💬 Discuss
- What patterns do you see: gender, age, skin tone, body type, clothing, setting? Which rows look the most alike?
- Where could these patterns come from? Who takes photos, writes captions, and posts them online?
- Does it matter? Think about AI images in ads, stock photos, news articles, or school materials.
- Try *a female CEO* or *an elderly nurse*. Does changing the prompt fix the problem, or just hide it?
- Some companies quietly rewrite your prompt to add diversity. What could go right, and what could go wrong?

*This is one model (DreamShaper 8, built on Stable Diffusion 1.5). Other models show different patterns, but rarely none.*

In [ ]:
#@title ▶ Run the bias experiment
things_to_test = "a CEO, a nurse, a software engineer, a housekeeper" #@param ["a CEO, a nurse, a software engineer, a housekeeper", "a doctor, a teacher, a pilot, a cleaner", "a scientist, an artist, an athlete, a politician", "a person, a family, a wedding, a house"] {allow-input: true}
images_per_row = 4 #@param {type:"slider", min:2, max:6, step:1}
#@markdown Each item becomes the prompt *"a photo of …"*. Up to 5 items. Takes about 30–60 seconds.
if "lab" not in globals():
    print("⚠️ Please run ⚙️ 1. Setup first (scroll up), then run this cell again.")
else:
    lab.run_bias(things_to_test, images_per_row)

## 🩹 10. Bonus: Inpainting (replace part of an image)
Inpainting is image-to-image **inside a box**: only the area you choose gets noise and gets repainted, and the rest of the picture stays untouched. Photo apps use the same idea for "magic eraser" and "generative fill" tools.

**Step A:** choose the picture and move the sliders to place the red box. Press ▶ as often as you like; it's instant.
**Step B:** describe what should appear in the box and press ▶. (The first time takes about 1 extra minute to download the inpainting model.)

*"last generated image"* means your latest picture from section 2 or the playground.

In [ ]:
#@title ▶ Step A: choose the area
source = "last generated image" #@param ["last generated image", "my photo (asks to upload the first time)", "upload a new photo"]
x = 30 #@param {type:"slider", min:0, max:100, step:1}
y = 35 #@param {type:"slider", min:0, max:100, step:1}
width = 40 #@param {type:"slider", min:5, max:100, step:1}
height = 45 #@param {type:"slider", min:5, max:100, step:1}
#@markdown x / y = where the box starts (% from the left / top). width / height = box size (% of the picture).
if "lab" not in globals():
    print("⚠️ Please run ⚙️ 1. Setup first (scroll up), then run this cell again.")
else:
    lab.run_inpaint_area(source, x, y, width, height)

In [ ]:
#@title ▶ Step B: repaint the red area
prompt = "a giant teddy bear" #@param {type:"string"}
seed = 1 #@param {type:"integer"}
if "lab" not in globals():
    print("⚠️ Please run ⚙️ 1. Setup first (scroll up), then run this cell again.")
else:
    lab.run_inpaint(prompt, seed)

## 🎮 11. Bonus: Playground
All the knobs in one place. A few extras:
- **seed = -1** picks a random seed. The seed is printed, so you can get the image back.
- **sampler**: different math recipes for removing noise. Same seed + a different sampler → similar but not identical images. *Euler a* adds a little fresh randomness at every step, so changing the step count changes its image more.
- **size**: this model learned on 512×512 pictures. Try **1024 × 1024** and look for doubled heads or repeated objects, because the model doesn't know how to fill that much space.
- Negative prompts only work when CFG is above 1.

🏆 **Challenge:** your instructor shows an AI image. Which team can get closest to it?

In [ ]:
#@title ▶ Playground
prompt = "an astronaut riding a horse on the moon, highly detailed, cinematic lighting" #@param {type:"string"}
negative_prompt = "blurry, low quality, deformed" #@param {type:"string"}
seed = -1 #@param {type:"integer"}
steps = 20 #@param {type:"slider", min:1, max:50, step:1}
cfg = 7.0 #@param {type:"slider", min:1, max:20, step:0.5}
size = "512 x 512 (square)" #@param ["512 x 512 (square)", "768 x 512 (landscape)", "512 x 768 (portrait)", "1024 x 1024 (too big for this model: try it!)"]
sampler = "DPM++ 2M Karras" #@param ["DPM++ 2M Karras", "Euler a", "Euler", "DDIM", "DEIS"]
how_many = 2 #@param {type:"slider", min:1, max:4, step:1}
if "lab" not in globals():
    print("⚠️ Please run ⚙️ 1. Setup first (scroll up), then run this cell again.")
else:
    lab.run_playground(prompt, negative_prompt, seed, steps, cfg, size, sampler, how_many)

## 🧠 What did we learn?

| Knob | What it really does |
|---|---|
| **Prompt** | steers every denoising step toward a description |
| **Seed** | picks the starting static, so it makes results reproducible |
| **Steps** | how many clean-ups; quality levels off while cost keeps growing |
| **CFG** | how hard to push toward the prompt (vs. no prompt) |
| **Negative prompt** | what to push away from |
| **Strength** (img2img) | how much of your photo gets buried in noise and repainted |
| **Mask** (inpainting) | where the AI is allowed to repaint |

### Bigger questions to take with you
- **Training data and consent.** The model learned from images that artists and photographers posted online, mostly without being asked. Is that fair? Should they be paid?
- **Copyright.** Who owns an AI image: you, the model maker, nobody? (The answer differs by country and is still changing.)
- **Deepfakes.** You just edited a photo in seconds. What does that mean for photos as evidence, and for consent?
- **Labeling.** Should AI images always be labeled? Look up *content credentials* / *C2PA*.
- **Energy.** You saw the seconds per image on one GPU. Now imagine millions of users doing this all day.

---
### 🧑‍🏫 Instructor only
Builds the example gallery shown to students whose notebook has no GPU. Run it once on a T4 **after** setting `GALLERY_BASE_URL` in the Setup cell. It downloads `gallery.zip`; unzip it into the repo's `gallery/` folder and push.

In [ ]:
#@title 🧑‍🏫 Build the backup gallery (instructor only)
export_now = False #@param {type:"boolean"}
#@markdown Tick the box, then press ▶. Re-runs every experiment with its default settings (~3–4 min on a T4).
if not export_now:
    print("Instructor only: tick export_now first.")
elif "lab" not in globals():
    print("⚠️ Please run ⚙️ 1. Setup first (scroll up), then run this cell again.")
else:
    lab.export_gallery()